# Probabilistic Tractography

Instead of always following the single most likely direction, probabilistic tractography **samples from the uncertainty** in the diffusion model. Each seed generates multiple streamlines, each taking a slightly different path. The result is a **path probability distribution** rather than a single tract.

## Why probabilistic?

1. **Noise**: the measured signal is noisy → the estimated direction has uncertainty
2. **Crossing fibres**: a voxel might contain two populations — probabilistic sampling can follow both
3. **Confidence quantification**: path probability = how consistently streamlines travel between A and B

## The two leading implementations

| Tool | Model | Key method |
|---|---|---|
| **FSL bedpostX + probtrackx2** | Ball-and-sticks (up to 3 sticks per voxel) | Bayesian inference on fibre orientations; sample from posterior |
| **MRtrix3 tckgen iFOD2** | Continuous CSD FOD | Importance sampling on the FOD amplitude |
| **DIPY** | CSD + probabilistic getter | Sample from FOD amplitude distribution |

**iFOD2** (Improved FOD-based probabilistic streamlines tractography, Tournier 2010) is the current state-of-the-art for whole-brain probabilistic tractography and is the default for MRtrix3 pipelines.

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../../scripts')
from utils import run

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir = Path('../../data/hcp/100307/preprocessed')
csd_dir  = Path('../../data/hcp/100307/csd')
tck_dir  = Path('../../data/hcp/100307/tractography')
tck_dir.mkdir(parents=True, exist_ok=True)

mask        = str(data_dir / 'nodif_brain_mask.nii.gz')
wm_fod_norm = str(csd_dir / 'wmfod_norm.mif')

print('Setup complete.')

## Approach A: MRtrix3 tckgen iFOD2 (recommended)

iFOD2 is the recommended algorithm for whole-brain tractography. It uses the full FOD amplitude (not just peaks) to weight the probability of each tracking direction.

In [ ]:
# ─── [MRtrix3] tckgen iFOD2 — whole-brain tractography ───────────────────────
#
# We request 10 million streamlines (standard for HCP).
# Reduce to 1M for a quick test.
#
# -act: Anatomically Constrained Tractography (ACT). Requires 5-tissue-type image.
#       We show how to generate it from the T1, but use seed_image=mask here
#       for simplicity.
#
# -backtrack: if a streamline gets stuck, back up and try a different direction

prob_tck_mrt = str(tck_dir / 'prob_iFOD2_1M.tck')

mrt_prob_cmd = [
    'tckgen',
    wm_fod_norm,
    prob_tck_mrt,
    '-algorithm', 'iFOD2',
    '-seed_image', mask,
    '-select', '1000000',    # 1M streamlines
    '-minlength', '10',
    '-maxlength', '250',
    '-angle', '45',
    '-step', '0.5',
    '-backtrack',
    '-force',
    '-nthreads', '4',
]

print('[MRtrix3] tckgen iFOD2 command:')
print(' '.join(mrt_prob_cmd))
print()
print('>> Runtime: ~5-10 min for 1M streamlines on a modern CPU')
print('>> Uncomment to run:')

# result = subprocess.run(mrt_prob_cmd, capture_output=True, text=True)
# print(result.stdout[-2000:])

## Add ACT: Anatomically Constrained Tractography

ACT dramatically improves tractography quality by using the T1w image to constrain where streamlines can go. This requires a **5-tissue-type (5TT) image**.

In [ ]:
# ─── [MRtrix3] 5TT image generation ──────────────────────────────────────────
#
# The 5TT image has 5 volumes:
#   0: cortical grey matter
#   1: subcortical grey matter
#   2: white matter
#   3: CSF
#   4: pathological tissue

t1_brain = str(data_dir.parent / 'T1w_acpc_dc_restore_brain.nii.gz')
ftt_image = str(prep_dir / '5tt.mif')

ftt_cmd = [
    '5ttgen', 'fsl',        # uses FSL FAST for tissue segmentation
    t1_brain,
    ftt_image,
    '-force',
]
print('[MRtrix3] 5TTgen command (requires FSL FAST):')
print(' '.join(ftt_cmd))
print()

# result = subprocess.run(ftt_cmd, capture_output=True, text=True)
# print(result.stdout or '(done)')

# With 5TT, tckgen command adds -act and -crop_at_gmwmi flags:
prob_tck_act = str(tck_dir / 'prob_iFOD2_ACT_1M.tck')
mrt_act_cmd = [
    'tckgen', wm_fod_norm, prob_tck_act,
    '-algorithm', 'iFOD2',
    '-act', ftt_image,          # anatomical constraint
    '-backtrack',
    '-crop_at_gmwmi',           # end streamlines at GM-WM interface
    '-select', '1000000',
    '-force', '-nthreads', '4',
]
print('[MRtrix3] tckgen with ACT command:')
print(' '.join(mrt_act_cmd))
print()
print('ACT benefits:')
print('  • Streamlines cannot end in WM (eliminates premature termination)')
print('  • Streamlines cannot re-enter CSF (eliminates anatomically implausible paths)')
print('  • Much cleaner tractogram, especially at cortical boundaries')

## Approach B: DIPY probabilistic tractography

In [ ]:
# ─── [DIPY] Probabilistic tractography ───────────────────────────────────────
from dipy.io.gradients import read_bvals_bvecs
from dipy.core.gradients import gradient_table
from dipy.reconst.csdeconv import ConstrainedSphericalDeconvModel, auto_response_ssst
from dipy.direction import ProbabilisticDirectionGetter, peaks_from_model
from dipy.tracking.local_tracking import LocalTracking
from dipy.tracking.streamline import Streamlines
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion
from dipy.tracking import utils as tracking_utils
from dipy.reconst.dti import TensorModel, fractional_anisotropy
from dipy.data import get_sphere
from dipy.io.stateful_tractogram import StatefulTractogram, Space
from dipy.io.streamline import save_trk

# Load
bvals, bvecs = read_bvals_bvecs(
    str(data_dir / 'bvals'), str(data_dir / 'bvecs'))
gtab = gradient_table(bvals, bvecs)

img   = nib.load(str(data_dir / 'data.nii.gz'))
data  = img.get_fdata()
maskd = nib.load(mask).get_fdata().astype(bool)

sel   = (bvals < 50) | ((bvals > 900) & (bvals < 1100))
gtab1 = gradient_table(bvals[sel], bvecs[sel])
d1    = data[..., sel]

# Fit CSD
response, _ = auto_response_ssst(gtab1, d1, roi_radii=10, fa_thr=0.7)
csd_model   = ConstrainedSphericalDeconvModel(gtab1, response, sh_order=8)

sphere = get_sphere('symmetric724')
csd_peaks = peaks_from_model(
    model=csd_model, data=d1, sphere=sphere,
    relative_peak_threshold=0.5, min_separation_angle=25,
    mask=maskd, npeaks=3, normalize_peaks=True, return_sh=True,
)

# Stopping criterion
stopping_criterion = ThresholdStoppingCriterion(csd_peaks.gfa, 0.25)

# Probabilistic direction getter: samples from FOD amplitude
prob_getter = ProbabilisticDirectionGetter.from_shcoeff(
    csd_peaks.shm_coeff,
    max_angle=45.0,
    sphere=sphere,
)

# Seeds from FA mask
tenfit = TensorModel(gtab1, fit_method='WLS').fit(d1, mask=maskd)
FA     = fractional_anisotropy(tenfit.evals)
seeds  = tracking_utils.seeds_from_mask(FA > 0.2, affine=img.affine, density=2)
print(f'Seeds: {len(seeds):,}')

# Run probabilistic tractography
print('Running DIPY probabilistic tractography ...')
streamline_generator = LocalTracking(
    direction_getter=prob_getter,
    stopping_criterion=stopping_criterion,
    seeds=seeds,
    affine=img.affine,
    step_size=0.5,
    max_cross=1,
)
streamlines = Streamlines(streamline_generator)

# Filter by length
lengths = np.array([len(s) * 0.5 for s in streamlines])
valid   = (lengths >= 10) & (lengths <= 250)
streamlines = streamlines[valid]

print(f'✓ {len(streamlines):,} streamlines')

dipy_prob_trk = str(tck_dir / 'prob_dipy.trk')
save_trk(StatefulTractogram(streamlines, img, Space.RASMM), dipy_prob_trk)
print(f'Saved: {dipy_prob_trk}')

## Approach C: FSL bedpostX + probtrackx2

FSL uses a Bayesian model (ball-and-sticks) to estimate fibre populations and their uncertainty. This is then used by `probtrackx2` to propagate streamlines by sampling from the posterior.

In [ ]:
# ─── [FSL] bedpostX + probtrackx2 ─────────────────────────────────────────────
#
# bedpostX fits a ball-and-sticks model at every voxel using MCMC.
# The output is a directory of samples from the posterior distribution
# of fibre orientations.

# For this to work, the input directory needs:
#   data.nii.gz   — eddy-corrected DWI
#   bvals, bvecs  — gradient table (rotated bvecs!)
#   nodif_brain_mask.nii.gz — binary brain mask

bedpostx_input = str(prep_dir)
bedpostx_out   = bedpostx_input + '.bedpostX'

bpx_cmd = [
    'bedpostx',
    bedpostx_input,
    '--nf=3',          # 3 fibre populations
    '--fudge=1',
    '--bi_convex',     # better convergence with crossing fibres
    '--rician',        # Rician noise model (more accurate at low SNR)
]
print('[FSL] bedpostX command:')
print(' '.join(bpx_cmd))
print()
print('>> bedpostX runtime: ~6-12 hours on CPU, ~1-2 hours on GPU')
print('>> Use bedpostx_gpu for GPU acceleration')
print()

# After bedpostX, run probtrackx2
seed_mask = str(data_dir / 'nodif_brain_mask.nii.gz')
target_mask = seed_mask   # whole-brain; can be an ROI mask

ptrackx_cmd = [
    'probtrackx2',
    '-s', f'{bedpostx_out}/merged',      # samples from bedpostX
    '-m', str(data_dir / 'nodif_brain_mask.nii.gz'),
    '-x', seed_mask,
    '--dir=' + str(tck_dir / 'fsl_probtrackx'),
    '--loopcheck',                        # stop if streamline loops
    '--onewaycondition',
    '--nsamples=5000',                    # samples per seed
    '--nsteps=2000',                      # max steps per sample
    '--steplength=0.5',
    '--distthresh=10',
    '--opd',                              # output path distribution
]
print('[FSL] probtrackx2 command (run after bedpostX):')
print(' '.join(ptrackx_cmd))

## Compare path distributions

In [ ]:
# Compare DIPY deterministic vs probabilistic tractography density
from dipy.io.streamline import load_trk
from dipy.tracking.utils import density_map

def streamline_density(trk_path, ref_img):
    sft = load_trk(trk_path, ref_img)
    dm  = density_map(sft.streamlines, affine=ref_img.affine,
                      vol_dims=ref_img.shape[:3])
    return dm

det_trk  = str(tck_dir / 'det_dipy.trk')
prob_trk = dipy_prob_trk

if Path(det_trk).exists() and Path(prob_trk).exists():
    det_dm  = streamline_density(det_trk,  img)
    prob_dm = streamline_density(prob_trk, img)

    z = data.shape[2] // 2
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, dm, label in zip(axes,
                              [det_dm, prob_dm],
                              ['Deterministic', 'Probabilistic']):
        ax.imshow(np.log1p(dm[:, :, z]).T, cmap='inferno', origin='lower')
        ax.set_title(f'DIPY {label}\n(log density)')
        ax.axis('off')

    fig.suptitle('Streamline density: deterministic vs probabilistic', fontsize=13)
    plt.tight_layout()
    plt.show()

    print('Observe:')
    print('  • Deterministic: narrower, more confident paths')
    print('  • Probabilistic: broader, captures uncertainty — especially in crossing regions')
else:
    print('Run both tractography approaches to compare.')

---

## Summary

| | MRtrix3 iFOD2 | DIPY Probabilistic | FSL bedpostX/probtrackx2 |
|---|---|---|---|
| **Model** | CSD FOD | CSD FOD | Ball-and-sticks (Bayesian) |
| **Sampling** | Importance sampling on FOD | Sample from FOD amplitude | MCMC posterior samples |
| **ACT support** | Yes (5ttgen) | Via custom stopping criteria | No native ACT |
| **Speed** | Fast (C++, parallel) | Moderate | Very slow (MCMC) |
| **Whole-brain** | Yes | Yes | Practical only for ROI-to-ROI |
| **Recommendation** | **Best for whole-brain** | **Best for learning/research** | **Best for ROI connectivity** |

**Next**: [Streamline filtering with SIFT →](03_filtering.ipynb)